In [12]:

class MOE(nn.Module):
    def __init__(self, n_dim, exps=2, topk=2):
        super(MOE, self).__init__()
        self.experts = nn.ModuleList([nn.Sequential(
            nn.Linear(n_dim, n_dim * 4),
            nn.ReLU(),
            nn.Linear(n_dim * 4, n_dim),
        ) for _ in range(exps)])

        self.exps = exps
        self.topk = topk

        self.router = nn.Linear(n_dim, exps)
        self.noise = nn.Linear(n_dim, exps)


    def forward(self, x):
        r = self.router(x)
        nl = self.noise(x)
        no = torch.randn_like(nl) * nl + r
        w, i = torch.topk(no, self.topk, dim=-1)
        b = torch.full_like(no, -1e6)
        s = torch.scatter(b, -1, i, w)
        go = torch.softmax(s, dim=-1)

        res = torch.zeros_like(x)


        for j, expert in enumerate(self.experts):
            em = (i == j).any(dim=-1)

            if em.any():
                expert_input = x[em]
                expert_output = expert(expert_input)

                gating_scores = go[em, j].unsqueeze(1)
                weighted_output = expert_output * gating_scores

                res[em] += weighted_output


        return res

class TL(nn.Module):
    def __init__(self, n_dim, n_head):
        super(TL, self).__init__()
        self.mha = RMHA(n_dim, n_head)
        self.ff = MOE(n_dim, exps=4, topk=2)

        self.n1 = nn.RMSNorm(n_dim)
        self.n2 = nn.RMSNorm(n_dim)

    def forward(self, x, m=None, freq=None, at=False):
        e = self.mha(self.n1(x), m, freq) + x
        e = self.ff(self.n2(x)) + e
        return e


class Transformer(nn.Module):
    def __init__(self, vocab_size, n_dim=16, n_head=4, layers=2, max_len=20):
        super(Transformer, self).__init__()
        self.emb = nn.Embedding(vocab_size, n_dim)

        self.tl = nn.ModuleList([TL(n_dim, n_head) for _ in range(layers)])
        self.n1 = nn.RMSNorm(n_dim)
        self.lm = nn.Sequential(
            nn.Linear(n_dim, vocab_size),
            nn.LogSoftmax(dim=-1)
        )
        self.n_dim = n_dim

        pa = torch.arange(0, max_len)
        freq_size = torch.arange(0, n_dim // n_head // 2)
        base_frecs = 1.0 / torch.pow(20, 2 * freq_size / n_dim)

        self.fx = torch.flatten(pa.unsqueeze(-1) * base_frecs, end_dim=1).unsqueeze(0)

    def forward(self, x, m=None):
        e = self.emb(x)
        for tl in self.tl:
            e = tl(e, m, self.fx)
        return self.lm(self.n1(e))

    def gen(self, s, max_len=10, bs=16):
        self.eval()

        ml = max([len(q) for q in s])
        at = [[1] * len(q) for q in s]
        s = ['-' * (ml - len(q)) + q for q in s]
        s = [[wi[q] for q in w] for w in s]
        at = [[0] * (ml - len(q)) + q for q in at]

        st = torch.LongTensor(s)
        at = torch.Tensor(at)

        res = []

        for x, m in DataLoader(TensorDataset(st, at), batch_size=bs, shuffle=False):
            for _ in range(max_len):
                with torch.no_grad():
                    o = self(x, m)
                    ne = o.exp().argmax(dim=-1)[:, -1].unsqueeze(1)
                    x = torch.cat([x, ne], dim=1)
                    m = torch.cat([m, torch.ones(m.shape[0], 1)], dim=1)
            res.append(x)

        fr = torch.cat(res, dim=0)
        fres = []
        for q in fr:
            fres.append(''.join([iw[c] for c in q.tolist()]))

        return fres


t = Transformer(vocab_size=len(wi), n_dim=64, n_head=8, layers=2, max_len=40)
x, m, y = data[0]
print(t(x.unsqueeze(0)).shape)